In [1]:
!pip install xgboost PyPDF2 reportlab pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 23.3 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

np.random.seed(42)

# ------------------------
# Customers Table
# ------------------------
num_customers = 2000

customers = pd.DataFrame({
    "customer_id": range(1, num_customers + 1),
    "age": np.random.randint(21, 70, num_customers),
    "annual_income": np.random.randint(200000, 2000000, num_customers),
    "risk_profile": np.random.choice(["low", "medium", "high"], num_customers, p=[0.6,0.3,0.1])
})

customers.to_json("customers.json", orient="records")

# ------------------------
# Device Intelligence XML
# ------------------------
import xml.etree.ElementTree as ET

root = ET.Element("devices")

for i in range(1, 5000):
    device = ET.SubElement(root, "device")
    ET.SubElement(device, "device_id").text = str(i)
    ET.SubElement(device, "device_risk_score").text = str(np.random.randint(1, 100))

tree = ET.ElementTree(root)
tree.write("device_data.xml")

# ------------------------
# Transactions CSV
# ------------------------
num_txns = 10000
start_time = datetime.now() - timedelta(days=30)

transactions = []

for i in range(num_txns):
    cust_id = random.randint(1, num_customers)
    amount = abs(np.random.normal(3000, 2000))
    txn_time = start_time + timedelta(minutes=random.randint(0, 43200))
    device_id = random.randint(1, 4999)

    # Fraud logic (synthetic)
    fraud = 1 if amount > 8000 and customers.loc[cust_id-1, "risk_profile"] == "high" else 0

    transactions.append([
        i+1, cust_id, device_id, round(amount,2), txn_time, fraud
    ])

transactions = pd.DataFrame(transactions,
                            columns=["transaction_id","customer_id","device_id","amount","timestamp","is_fraud"])

transactions.to_csv("transactions.csv", index=False)

print("Synthetic Data Generated")

Synthetic Data Generated


In [3]:
from reportlab.platypus import SimpleDocTemplate, Paragraph
from reportlab.lib.styles import getSampleStyleSheet

doc = SimpleDocTemplate("kyc_document.pdf")
styles = getSampleStyleSheet()
elements = []

elements.append(Paragraph("Customer KYC Risk Score: 72", styles["Normal"]))

doc.build(elements)

print("PDF Generated")

PDF Generated


In [4]:
# Load CSV
transactions = pd.read_csv("transactions.csv")

# Load JSON
customers = pd.read_json("customers.json")

# Load XML
import xml.etree.ElementTree as ET
tree = ET.parse("device_data.xml")
root = tree.getroot()

device_data = []
for device in root.findall("device"):
    device_data.append({
        "device_id": int(device.find("device_id").text),
        "device_risk_score": int(device.find("device_risk_score").text)
    })

device_df = pd.DataFrame(device_data)

# Load PDF
import PyPDF2
pdf = PyPDF2.PdfReader(open("kyc_document.pdf", "rb"))
kyc_text = pdf.pages[0].extract_text()

print("All Data Loaded Successfully")

All Data Loaded Successfully


In [5]:
# Null Check
print("Null values:\n", transactions.isnull().sum())

# Duplicate Check
print("Duplicates:", transactions.duplicated().sum())

# Schema Validation
expected_cols = ["transaction_id","customer_id","device_id","amount","timestamp","is_fraud"]
assert set(expected_cols).issubset(transactions.columns)

# Referential Integrity
missing_customers = transactions[~transactions['customer_id'].isin(customers['customer_id'])]
print("Missing Customers:", len(missing_customers))

# Business Rule Validation
invalid_amount = transactions[transactions['amount'] <= 0]
print("Invalid Amount Rows:", len(invalid_amount))

Null values:
 transaction_id    0
customer_id       0
device_id         0
amount            0
timestamp         0
is_fraud          0
dtype: int64
Duplicates: 0
Missing Customers: 0
Invalid Amount Rows: 0


In [6]:
df = transactions.merge(customers, on="customer_id", how="left")
df = df.merge(device_df, on="device_id", how="left")

df['timestamp'] = pd.to_datetime(df['timestamp'])

In [9]:
# Time-based feature
df['txn_hour'] = df['timestamp'].dt.hour

# Amount to income ratio
df['amt_income_ratio'] = df['amount'] / df['annual_income']

# Velocity feature
df = df.sort_values(['customer_id','timestamp'])

# Calculate rolling transaction count, ensuring the result is indexed by the original DataFrame's index
df['txn_count_10min'] = df.groupby('customer_id', group_keys=False).apply(
    lambda x: pd.Series(
        x.set_index('timestamp')['transaction_id'].rolling('10min').count().values,
        index=x.index # Assign the calculated values back to the original index of the group 'x'
    )
)

# Encode categorical
df = pd.get_dummies(df, columns=['risk_profile'], drop_first=True)

df.fillna(0, inplace=True)

/tmp/ipython-input-287/2351475496.py:11: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df['txn_count_10min'] = df.groupby('customer_id', group_keys=False).apply(


In [10]:
df.to_parquet("offline_feature_store.parquet")
print("Offline Feature Store Saved")

Offline Feature Store Saved


In [11]:
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score

X = df.drop(["is_fraud","timestamp"], axis=1)
y = df["is_fraud"]

X_train, X_test, y_train, y_test = train_test_split(X,y,stratify=y,test_size=0.2)

model = XGBClassifier(scale_pos_weight=10)
model.fit(X_train,y_train)

y_pred = model.predict(X_test)

print(classification_report(y_test,y_pred))
print("ROC AUC:", roc_auc_score(y_test, model.predict_proba(X_test)[:,1]))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1999
           1       0.00      0.00      0.00         1

    accuracy                           1.00      2000
   macro avg       0.50      0.50      0.50      2000
weighted avg       1.00      1.00      1.00      2000

ROC AUC: 1.0


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [12]:
import joblib
model_version = "v1"
joblib.dump(model, f"fraud_model_{model_version}.pkl")
print("Model Saved:", model_version)

Model Saved: v1


In [13]:
def predict_transaction(sample):
    model = joblib.load("fraud_model_v1.pkl")
    return model.predict(sample)

sample_txn = X_test.iloc[[0]]
print("Prediction:", predict_transaction(sample_txn))

Prediction: [0]


In [14]:
# Compare distribution between train and test
import scipy.stats as stats

for col in X_train.columns[:5]:
    stat, p_value = stats.ks_2samp(X_train[col], X_test[col])
    print(f"{col} drift p-value:", p_value)

transaction_id drift p-value: 0.041367655032781864
customer_id drift p-value: 0.014674807949598861
device_id drift p-value: 0.9151564989663031
amount drift p-value: 0.6988004415607497
age drift p-value: 0.7959604621148515


In [15]:
try:
    assert "amount" in transactions.columns
except:
    print("Schema mismatch detected!")

In [16]:
import time

start = time.time()
model.predict(sample_txn)
print("Inference time:", time.time()-start)

Inference time: 0.006717681884765625


In [17]:
import logging

logging.basicConfig(filename='model_logs.log', level=logging.INFO)
logging.info("Model prediction executed successfully.")